# バンディット問題勉強用ノートブック
このノートブックは、1状態のマルコフ決定過程（MDP）と形式的に等価なバンディット問題を説明するために作成されました。バンディット問題は、有限のアクション集合 $\mathcal{A}$ と、現在のアクションに対する報酬の確率分布 $p$ のタプル $(\mathcal{A}, p)$ で定義されます。

**注意点**

- `GaussianBandit` は教育用のシンプルなクラスです（`gymnasium.Env` のサブクラスではありません）。
- `epsilon_greedy_policy` は、Q値が等しい場合に特定のアクション（アクション0など）に偏らないよう、ランダムなタイブレーク（同値選択）を使用します。
- すべての確率的なルーチンは `seed` 引数を受け付け、複数シードの平均をとるための `run_experiment` を提供しています。

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from ipywidgets import interact, fixed
%matplotlib inline

# バンディット問題の例
5台のスロットマシンがあり、エージェントは5つの離散的な行動を選択できます。エージェントはアクションの選択に応じてスカラー報酬を受け取ります。目標は、期待報酬を最大化する行動を見つけることです。

In [ ]:
class GaussianBandit:
    """Simple multi-armed Gaussian bandit for educational use.

    This is intentionally NOT a gymnasium.Env subclass. For a bandit (one-state
    MDP) we only need a `step(action) -> reward` interface, and a concrete
    numpy-based random generator with a `seed` argument.
    """

    def __init__(self, seed=None):
        self.mu    = np.array([1.0, 5.0, 3.0, 2.0, 3.5])
        self.sigma = np.array([0.8, 0.9, 1.5, 1.5, 1.2])
        self.n_actions = len(self.mu)
        self.reset(seed=seed)

    # --- core API ---
    def reset(self, seed=None):
        self.rng = np.random.default_rng(seed)
        return None   # no observation (single state)

    def step(self, action):
        """Return a scalar reward drawn from N(mu[a], sigma[a])."""
        return float(self.rng.normal(self.mu[action], self.sigma[action]))

    # --- utilities ---
    @property
    def optimal_action(self):
        return int(np.argmax(self.mu))

    def plot(self):
        fig = plt.figure(figsize=(8, 5))
        axarr = fig.subplots(1, 1)
        x = np.arange(start=-3.0, stop=8.0, step=0.05)
        for index, mean in enumerate(self.mu):
            std = self.sigma[index]
            norm_pdf = stats.norm.pdf(x=x, loc=mean, scale=std)
            axarr.plot(x, norm_pdf, label='r%1.0f' % index)
        axarr.legend()
        axarr.set_xlabel('r')
        axarr.set_ylabel('pdf')
        axarr.grid()


## 報酬を与える確率分布の可視化
この例題ではガウス分布で報酬を確率的に生成しています。

In [ ]:
env = GaussianBandit(seed=0)
env.plot()
print('optimal action =', env.optimal_action)

## 一様ランダム方策

より洗練されたアクション選択手法を検討する前に、シンプルな一様ランダム方策でベースラインを確認しましょう。この手法では、エージェントは完全にランダムにアクションを選択します。つまり、利用可能な各アクションが等しい確率で選択されます。各アクションのQ値（期待報酬）は、そのアクションが選択されたときに受け取ったすべての報酬の算術平均として推定されます。

In [ ]:
def uniform_random_method(number_of_steps=1000, seed=None, show_plot=True):
    """Single-run bandit experiment using a uniform random policy.

    Parameters
    ----------
    number_of_steps : int
    seed : int or None
        Random seed that controls both the environment and action sampling.
    show_plot : bool
        If True, draw the per-step Q and N curves.

    Returns
    -------
    dict with keys:
        Q : (n_actions, T+1) array
        N : (n_actions, T+1) array
        rewards : (T,) array
        actions : (T,) int array
        optimal_action : int
    """
    env = GaussianBandit(seed=seed)
    rng = np.random.default_rng(None if seed is None else seed + 10_000)

    n_actions = env.n_actions

    # Initialize Q and N to store trajectories
    Q_trajectory = np.zeros((n_actions, number_of_steps + 1))
    N_trajectory = np.zeros((n_actions, number_of_steps + 1))

    # Variables to calculate arithmetic mean
    sum_rewards_per_action = np.zeros(n_actions)
    N_per_action = np.zeros(n_actions)

    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        # Uniformly random action selection
        action = int(rng.choice(n_actions))
        reward = env.step(action)

        # Update counts and sum of rewards for the chosen action
        N_per_action[action] += 1
        sum_rewards_per_action[action] += reward

        # Store counts for plotting
        N_trajectory[:, t+1] = N_per_action

        # Calculate Q(a) as arithmetic mean for all actions
        for i in range(n_actions):
            if N_per_action[i] > 0:
                Q_trajectory[i, t+1] = sum_rewards_per_action[i] / N_per_action[i]
            else:
                Q_trajectory[i, t+1] = 0.0 # Keep 0 if no rewards yet

        rewards[t] = reward
        actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q_trajectory[i, :], label='$a%i$' % i)
        axarr[0].legend()
        axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)')
        axarr[0].set_title('Q-values over steps (Uniform Random)')
        axarr[0].grid()

        for i in range(n_actions):
            axarr[1].plot(N_trajectory[i, :], label='$N%i$' % i)
        axarr[1].legend()
        axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)')
        axarr[1].set_title('Action Counts over steps (Uniform Random)')
        axarr[1].grid()
        plt.show()
        print('final Q:', Q_trajectory[:, -1])
        print('average reward: %f' % rewards.mean())

    return dict(Q=Q_trajectory, N=N_trajectory, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)

In [ ]:
_ = uniform_random_method(number_of_steps=1000, seed=0)

# Action selection
## $\varepsilon$-greedy policy
$\varepsilon$-greedy is a method to balance exploration and exploitation in the value-based reinforcement learning. When the current estimate of the action-value function is $Q(a)$, the probability to select the action $a$ is given by
\begin{equation}
  \pi (a) =
  \begin{cases}
    \epsilon/\lvert A \rvert + (1 - \epsilon)/|\mathrm{argmax}\,Q| & \mathrm{if} \; \;
       a \in \mathrm{arg}\max_a Q(a), \\
    \epsilon/\lvert A \rvert & \mathrm{otherwise},
  \end{cases}
\end{equation}
where $\varepsilon \in [0, 1]$ is a hyperparameter and $\lvert A \rvert$ is the number of available actions.

**Teaching note.** A naive `argmax` always returns the first index when there are ties. This means that when all $Q(a)=0$ at the start, action 0 is structurally favored. We therefore use `argmax_random_tie` below so that ties are broken uniformly at random.

In [ ]:
def argmax_random_tie(x, rng=None):
    """Return an argmax of x, breaking ties uniformly at random."""
    if rng is None:
        rng = np.random.default_rng()
    x = np.asarray(x)
    max_val = np.max(x)
    best = np.flatnonzero(x == max_val)
    return int(rng.choice(best))


def epsilon_greedy_policy(Q, epsilon=0.1):
    """Epsilon-greedy policy (returns a probability vector over actions).

    With random tie-breaking among argmax actions.
    """
    n_actions = Q.shape[0]
    policy = epsilon / n_actions * np.ones(n_actions)
    max_val = np.max(Q)
    best = np.flatnonzero(Q == max_val)
    policy[best] += (1 - epsilon) / len(best)
    return policy


def plot_epsilon_greedy_policy(Q, epsilon=0.1):
    policy = epsilon_greedy_policy(Q, epsilon)
    action = np.arange(Q.shape[0])

    fig = plt.figure(figsize=(12, 5))
    axarr = fig.subplots(1, 2)
    axarr[0].bar(action, Q)
    axarr[0].set_xlabel('action a')
    axarr[0].set_ylabel('action value Q(a)')

    axarr[1].bar(action, policy, label=r'$\epsilon$ = %3.1f' % epsilon)
    axarr[1].set_xlabel('action a')
    axarr[1].set_ylabel(r'policy $\pi$ (a)')

    plt.legend()
    plt.show()


In [ ]:
# epsilon = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}
Q = np.array([2.5, -2.5, 2.9, 1.2, 0.5])
interact(plot_epsilon_greedy_policy, Q=fixed(Q), epsilon=(0, 1, 0.05));


# 価値ベースの手法

エージェントは期待報酬を学習します。エージェントがアクション $a$ を選択し、スカラー報酬 $r$ を受け取ると、期待報酬の推定値は次のように更新されます。
\begin{equation*}
  Q(a) = Q(a) + \alpha (r - Q(a)),
\end{equation*}
ここで $\alpha$ は学習率です。以下では、各ステップでの $Q$ 推定値とアクションのカウントを記録するシングルランのルーチンを提供します。

In [ ]:
def value_based_method(method,
                        beta=1.0,
                        epsilon=0.1,
                        number_of_steps=1000,
                        initial_Q=0.0,
                        seed=None,
                        show_plot=True):
    """Single-run bandit experiment.

    Parameters
    ----------
    method : {"epsilon", "Boltzmann"}
    beta : float
        Inverse temperature for Boltzmann policy.
    epsilon : float
        Exploration rate for epsilon-greedy.
    number_of_steps : int
    initial_Q : float or ndarray
        Initial value of the action-value function. Set to a large positive
        value (e.g. 10) to enable *optimistic initialization*.
    seed : int or None
        Random seed that controls both the environment and action sampling.
    show_plot : bool
        If True, draw the per-step Q and N curves.

    Returns
    -------
    dict with keys:
        Q : (n_actions, T+1) array
        N : (n_actions, T+1) array
        rewards : (T,) array
        actions : (T,) int array
        optimal_action : int
    """
    env = GaussianBandit(seed=seed)
    rng = np.random.default_rng(None if seed is None else seed + 10_000)

    n_actions = env.n_actions
    Q = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))
    Q[:, 0] = initial_Q
    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        if method == 'epsilon':
            if epsilon == 0: # Pure greedy, use explicit random tie-breaking
                action = argmax_random_tie(Q[:, t], rng=rng)
            else:
                policy = epsilon_greedy_policy(Q[:, t], epsilon=epsilon)
                action = int(rng.choice(n_actions, p=policy))
        elif method == 'Boltzmann':
            policy = Boltzmann_policy(Q[:, t], beta=beta)
            action = int(rng.choice(n_actions, p=policy))
        else:
            raise ValueError(f'unknown method: {method}')
        reward = env.step(action)

        N[:, t+1] = N[:, t]
        N[action, t+1] += 1
        Q[:, t+1] = Q[:, t]
        alpha = 1.0 / N[action, t+1]
        Q[action, t+1] = Q[action, t] + alpha * (reward - Q[action, t])

        rewards[t] = reward
        actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q[i, :], label='$a%i$' % i)
        axarr[0].legend()
        axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)')
        axarr[0].set_title('Q-values over steps (epsilon-greedy policy)')
        axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend()
        axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)')
        axarr[1].grid()
        plt.show()
        print('final Q:', Q[:, -1])
        print('average reward: %f' % rewards.mean())

    return dict(Q=Q, N=N, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)


In [ ]:
# Generate a random seed for each execution
import numpy as np
current_run_seed = np.random.randint(0, 1_000_000)
epsilon = 0.05 #@param {type:"slider", min:0, max:1, step:0.05}
_ = value_based_method('epsilon', epsilon=epsilon, seed=current_run_seed)

## 複数回の実行による平均化と比較

1回だけの実行（シングルラン）はノイズが多く、最初の数回の試行が学習軌道を支配してしまいます。探索戦略を公平に評価するために、多くの独立したシードで平均をとり、以下の2つの標準的な曲線を確認します。

- **ステップごとの平均報酬**（アクションが平均してどの程度良いか）
- **最適アクション選択率**（最適な腕がどのくらいの頻度で選ばれているか）

また、以下の3つの設定を比較します。

1. $\varepsilon$-greedy ($\varepsilon=0.1$, $Q_0=0$) — 方策による探索
2. greedy ($\varepsilon=0$, $Q_0=10$) — **楽観的初期値**による探索
3. $\varepsilon=0.01$ (ベースライン)

In [ ]:
def run_experiment(method='epsilon',
                   n_runs=200,
                   number_of_steps=1000,
                   base_seed=0,
                   **kwargs):
    """Run `n_runs` independent bandit experiments and aggregate metrics.

    Returns
    -------
    avg_reward : (T,) array   -- per-step mean reward, averaged over runs
    opt_rate   : (T,) array   -- per-step optimal-action selection rate
    Q_last_run : (n_actions, T+1) array  -- Q trajectory of the last run
    """
    avg_reward_sum = None
    opt_action_sum = None
    last = None
    for k in range(n_runs):
        result = value_based_method(method,
                                    number_of_steps=number_of_steps,
                                    seed=base_seed + k,
                                    show_plot=False,
                                    **kwargs)
        if avg_reward_sum is None:
            avg_reward_sum = np.zeros(number_of_steps)
            opt_action_sum = np.zeros(number_of_steps)
        avg_reward_sum += result['rewards']
        opt_action_sum += (result['actions'] == result['optimal_action']).astype(float)
        last = result
    avg_reward = avg_reward_sum / n_runs
    opt_rate   = opt_action_sum / n_runs
    return avg_reward, opt_rate, last['Q']


def optimal_action_rate(actions, optimal_action):
    """Cumulative optimal-action selection rate."""
    hit = (np.asarray(actions) == optimal_action).astype(float)
    return np.cumsum(hit) / np.arange(1, len(hit) + 1)


In [ ]:
import matplotlib.pyplot as plt

# Compare three strategies, averaged over n_runs seeds
n_runs = 200
T = 1000

configs = [
    dict(label=r'$\varepsilon$=0.1, $Q_0$=0',  method='epsilon', epsilon=0.1,  initial_Q=0.0),
    dict(label=r'$\varepsilon$=0.01, $Q_0$=0', method='epsilon', epsilon=0.01, initial_Q=0.0),
    dict(label=r'greedy, $Q_0$=10 (optimistic)', method='epsilon', epsilon=0.0,  initial_Q=10.0),
]

results = []
for cfg in configs:
    current_cfg = cfg.copy()
    label = current_cfg.pop('label')
    avg_r, opt_r, Q_last = run_experiment(n_runs=n_runs, number_of_steps=T, **current_cfg)
    results.append((label, avg_r, opt_r, Q_last))

fig, axarr = plt.subplots(1, 2, figsize=(13, 5))
for label, avg_r, opt_r, _ in results:
    axarr[0].plot(avg_r, label=label)
    axarr[1].plot(opt_r, label=label)
axarr[0].set_xlabel('steps'); axarr[0].set_ylabel('average reward'); axarr[0].grid(); axarr[0].legend()
axarr[1].set_xlabel('steps'); axarr[1].set_ylabel('optimal action rate'); axarr[1].grid(); axarr[1].legend()
plt.tight_layout(); plt.show()

# And the per-arm Q trajectory of the last run (epsilon=0.1 run)
_, _, Q_last = results[0][1], results[0][2], results[0][3]
fig = plt.figure(figsize=(8, 5))
for i in range(Q_last.shape[0]):
    plt.plot(Q_last[i, :], label='$a%i$' % i)
plt.xlabel('steps'); plt.ylabel(r'Q(a)  (last run of $\varepsilon$=0.1)')
plt.grid(); plt.legend(); plt.show()